# 04c · Cost (C4)

**Purpose.** Feature size, time to build features, time to train the best head and GPU need, per representation and dataset.

**Inputs**
- Feature builders from `src/features/`, both datasets; best heads from `04a`

**Outputs**
- `results/04c_cost.csv`
- `report/tables/` cost table (D = 7200 / 864 / 2304 for GWT, 770 / 132 / 352 for ES)

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
import time

import torch
from src.cv import load_baseline_level, load_processed
from src.features.handcrafted import handcrafted_features
from src.features.raw import raw_features

TABLES = ROOT / "report/tables"
RESULTS = ROOT / "results"
MAIN = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
        "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
missing = [n for n in MAIN if not (RESULTS / f"{n}.csv").exists()]
assert not missing, f"run these notebooks first: {missing}"
scores = pd.concat([pd.read_csv(RESULTS / f"{n}.csv") for n in MAIN], ignore_index=True)
REPS = ["R1_raw", "R2_handcrafted", "R3_encoder"]
HEADS = ["logreg", "svm", "rf", "mlp"]

## C4: cost of each representation
- **feature size**: D
- **time to build features**: R1 and R2 are timed here on the full dataset; for R3 it is the per-fold
  encoder training + encoding time recorded by `run_cv` (`feat_seconds`)
- **time to train the best head**: mean `fit_seconds` (grid search included) of the best head
- **GPU**: whether the representation needs one

In [ ]:
build_time = {}
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)
    levels = load_baseline_level(dataset)
    t0 = time.perf_counter(); raw_features(X, 100 if dataset == "gwt" else 70); build_time[(dataset, "R1_raw")] = time.perf_counter() - t0
    t0 = time.perf_counter(); handcrafted_features(X, levels, info["n_base"], info["exposure_end"]); build_time[(dataset, "R2_handcrafted")] = time.perf_counter() - t0

rows = []
main = scores[scores["task"] == "multi"]
for (dataset, rep), s in main[main["rep"].isin(REPS)].groupby(["dataset", "rep"]):
    head = s.groupby("head")["macro_f1"].mean().idxmax()
    rows.append({
        "dataset": dataset, "rep": rep, "D": int(s["n_features"].iloc[0]),
        "build_features_s": build_time.get((dataset, rep), s["feat_seconds"].mean()),
        "best_head": head,
        "train_best_head_s": s[s["head"] == head]["fit_seconds"].mean(),
        "gpu": "optional (trains on CPU)" if rep == "R3_encoder" else "no",
    })
cost = pd.DataFrame(rows).round(2)
cost.to_csv(TABLES / "C4_cost.csv", index=False)
print("GPU available on this machine:", torch.cuda.is_available())
cost